In [1]:
import os

# Use conda's newer libstdc++ instead of the system one
conda_lib = os.path.expanduser("~/.conda/envs/hnn-core-env/lib")
ld_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_path}"

# UCX settings (from before)
os.environ["UCX_TLS"] = "tcp,self,sm"
os.environ["UCX_NET_DEVICES"] = "all"


# MPI paths
mpi_bin = "/oscar/rt/sw/external/hpcx-mpi/2.25.1/bin"
mpi_lib = "/oscar/rt/sw/external/hpcx-mpi/2.25.1/lib"
if mpi_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{mpi_bin}:{os.environ['PATH']}"
if mpi_lib not in ld_path:
    os.environ["LD_LIBRARY_PATH"] = f"{mpi_lib}:{os.environ['LD_LIBRARY_PATH']}"

import sys
# Add MPI bin directory to PATH
mpi_bin = "/oscar/rt/sw/external/hpcx-mpi/2.25.1/bin"
if mpi_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{mpi_bin}:{os.environ['PATH']}"

print("Updated PATH to include MPI")
print("MPI location:", mpi_bin)

import os
print("PATH:", os.environ.get('PATH', 'NOT SET'))
print("Which mpiexec:")
import subprocess
result = subprocess.run(['which', 'mpiexec'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "mpiexec not found")

In [ ]:
import os
import os.path as op
import numpy as np
import matplotlib.pyplot as plt

import hnn_core
from hnn_core.externals.mne import tfr_array_morlet
from hnn_core.viz import plot_tfr_morlet
from hnn_core import MPIBackend, JoblibBackend, average_dipoles

import sys
sys.path.append('/users/kduecker/hnn-tuning/new_model_HNN/')
sys.path.append('/users/kduecker/hnn-tuning/utils')
from general_plot_utils import assign_color

from hnn_core import duecker_ET_model, simulate_dipole
import matplotlib.pyplot as plt
import numpy as np
from hnn_core.externals.mne import tfr_array_morlet
pathfig = '/users/kduecker/hnn-tuning/new_model_HNN/figures/gamma'
os.makedirs(pathfig, exist_ok=True)

import matplotlib as mpl
import decimal
mpl.rcParams['pdf.fonttype'] = 42
n_procs=50

In [3]:
def _nice_ylim(y_max):
    """(upper, ticks) for axis from 0 to just above y_max.
    
    Rounds up to first significant digit. 3 ticks if that digit is even,
    4 if odd.
    """
    if not np.isfinite(y_max) or y_max <= 0:
        return 1.0, np.linspace(0, 1, 4)
    order = np.floor(np.log10(y_max))
    leading = int(np.ceil(y_max / 10**order))
    upper = leading * 10**order
    n_ticks = 3 if leading % 2 == 0 else 4
    return upper, np.linspace(0, upper, n_ticks)


def plot_single_trial_sim(net, dpl):

    fig = plt.figure(figsize=(20,8))

    gs = fig.add_gridspec(5,7, height_ratios=[.25, 0.25, 0.25, 0.25, 1.25], hspace=0.5, wspace=0.6)
    axempty = fig.add_subplot(gs[1:4,:1])
    axempty.set_axis_off()
    ax00 = fig.add_subplot(gs[0,1:3])
    ax10 = fig.add_subplot(gs[1:4,1:3])
    ax11 = fig.add_subplot(gs[0:2,3:5])
    ax21 = fig.add_subplot(gs[2:4,3:5])
    ax3 = fig.add_subplot(gs[:4,5:])
    ax4 = fig.add_subplot(gs[4,:2])
    ax5 = fig.add_subplot(gs[4, 2])
    ax6 = fig.add_subplot(gs[4, 3])
    ax7 = fig.add_subplot(gs[4, 4])
    ax8 = fig.add_subplot(gs[4, 5])
    # ax9 = fig.add_subplot(gs[4, 4:6])
    # axreal10 = fig.add_subplot(gs[4, 6:])

    # drives
    bins = np.arange(0, 1000, 5)
    spike_prox1 = np.array(net.cell_response.spike_times_by_type['prox_poiss'][0]).reshape(-1,1)

    ax00.hist(spike_prox1, bins=bins, color='r')
    ax00.set_xlim([0, 1000])
    ax00.set_xticks(np.arange(0, 1001, 250))

    ax00.set_xticklabels([])            
    # dipole
    ax10.plot(dpl[0].times, dpl[0].data['agg'], color='k', linewidth=1)
    ax10.set_ylabel('diple (nA·m)')
    ax10.set_xlim([0, 1000])
    ax10.spines['top'].set_visible(False)
    ax10.spines['right'].set_visible(False)
    ax10.set_xlabel('time (ms)')
    ax10.set_xticks(np.arange(0, 1001, 250))
    ax10.set_ylim([np.round(np.min(dpl[0].data['agg'])-0.01,2), np.round(np.max(dpl[0].data['agg'])+0.01,2)])
    ax10.set_yticks(np.linspace(np.round(np.min(dpl[0].data['agg']),2), np.round(np.max(dpl[0].data['agg'])+0.01,2),4))
    # dipole and raster
    I = np.array(net.cell_response.spike_gids[0]) < 135
    gids = np.array(net.cell_response.spike_gids[0]).reshape(-1,)[I]
    times = np.array(net.cell_response.spike_times[0]).reshape(-1,)[I] 
    colors = [assign_color(net, gid) for gid in gids]

    ax11.plot(dpl[0].times, dpl[0].data['L2'], color='k', alpha=0.5)
    ax11.set_ylim([0, 0.01])
    ax11.set_xlim([0, 1000])
    ax11.set_xticks(np.arange(0, 1001, 250))
    ax11.set_ylim([np.round(np.min(dpl[0].data['L2']-0.01),2), np.round(np.max(dpl[0].data['L2']+0.01),2)])
    ax11.set_yticks(np.linspace(np.round(np.min(dpl[0].data['L2'])-0.01,2), np.round(np.max(dpl[0].data['L2'])+0.01,2),4))
    ax11.set_ylabel('dipole (nA·m)')
    ax12 = ax11.twinx()

    ax12.scatter(times, gids, s=15, alpha=1., c=colors, marker='|')
    ax12.set_ylabel('L2/3 neurons')
    ax12.set_xticklabels([])
    ax12.invert_yaxis()
    ax12.set_yticks([0, 35, 135])
    ax12.set_xticks(np.arange(0, 1001, 250))


    I = (np.array(net.cell_response.spike_gids[0]) >= 135) & (np.array(net.cell_response.spike_gids[0]) < 270)
    gids = np.array(net.cell_response.spike_gids[0]).reshape(-1,)[I]
    times = np.array(net.cell_response.spike_times[0]).reshape(-1,)[I] 
    colors = [assign_color(net, gid) for gid in gids]

    ax21.plot(dpl[0].times, dpl[0].data['L5'], color='k', alpha=0.5)
    ax21.set_xlim([0, 1000])
    ax21.set_ylim([np.round(np.min(dpl[0].data['L5']-0.01),2), np.round(np.max(dpl[0].data['L5']+0.01),2)])
    ax21.set_yticks(np.linspace(np.round(np.min(dpl[0].data['L5']),2), np.round(np.max(dpl[0].data['L5'])+0.01,2),4))
    ax21.set_xticks(np.arange(0, 1001, 250))

    ax22 = ax21.twinx()
    ax22.scatter(times, gids, s=15, alpha=1., c=colors, marker='|')
    ax22.set_ylabel('L5 neurons')
    ax22.invert_yaxis()
    ax22.set_yticks([135, 170, 270])
    ax22.set_xlabel('time (ms)')
    ax22.set_xticks(np.arange(0, 1001, 250))

    # TFR
    freqs = np.arange(20, 81, 1)

    tfr = np.zeros((len(dpl), len(freqs), len(dpl[0].times)))

    for i,d in enumerate(dpl):

        cur_tfr = tfr_array_morlet(d.data["agg"][None, None, :], d.sfreq, freqs, output='power')
        tfr[i] = cur_tfr.squeeze()

    tfr = tfr.mean(axis=0)
    
    im = ax3.pcolormesh(dpl[0].times, freqs, tfr, shading='auto', cmap='RdBu_r', vmin=0, vmax=np.round(tfr.max()+0.02,1), rasterized=True)
    cb = plt.colorbar(im, pad=0.025, orientation='horizontal', shrink=0.4)
    cb.set_ticks([0, np.round(tfr.max()+0.02,1)])
    cb.set_label('power (nA·m)^2')
    ax3.set_xlabel('time (ms)')
    ax3.set_yticks([20, 40, 60, 80])
    ax3.set_ylabel('frequency (Hz)')
    ax3.set_xticks(np.arange(0, 1001, 250))
    # PSD
    from scipy.signal import periodogram
    freqs, Pxx = periodogram(dpl[0].data['agg'], dpl[0].sfreq, window="hamming", nfft=len(dpl[0].data['agg']))
    freqs, PxxL2 = periodogram(dpl[0].data['L2'], dpl[0].sfreq, window="hamming", nfft=len(dpl[0].data['L2']))
    freqs, PxxL5 = periodogram(dpl[0].data['L5'], dpl[0].sfreq, window="hamming", nfft=len(dpl[0].data['L5']))
    ax4tw = ax4.twinx()
    ax4tw.plot(freqs, PxxL2, color=np.array([164,30,79])/255, linewidth=2)
    
    ax4.plot(freqs, Pxx, color='k', linewidth=2)
    ax4.plot(freqs, PxxL5, color=np.array([92,114,183])/255, linewidth=2)
    ax4.set_xlim([20, 80])
    ax4.text(freqs[np.argmax(Pxx)], np.max(Pxx), f'{np.round(freqs[np.argmax(Pxx)]):.0f}Hz')
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)
    ax4.set_xlabel('frequency (Hz)')
    ax4.set_xticks([20,40,60,80])
    upper, ticks = _nice_ylim(Pxx.max())
    ax4.set_ylim([0, upper])
    ax4.set_yticks(ticks)

    upper_L2, ticks_L2 = _nice_ylim(PxxL2.max())
    ax4tw.set_ylim([0, upper_L2])
    ax4tw.set_yticks(ticks_L2)


    # calcium

    l23_cols_tone = ['#110308', '#420d20', '#741638', '#a52050']

    U, C = np.unique(np.array(net.cell_response.spike_gids[0])[(np.array(net.cell_response.spike_gids[0]) < 135) & (np.array(net.cell_response.spike_gids[0]) >= 35)], return_counts=True)
    gid_oi = U[np.argmax(C)]

    sections = ['soma', 'apical_trunk', 'apical_1', 'apical_tuft']

    L = []
    for section in sections:
        L.append(net.cell_types["L2_pyramidal"]['cell_object'].sections[section].L)
    prevL = 0
    for i, section in enumerate(sections):

        v = np.array(net.cell_response.vsec[0][gid_oi][section])
        ca = np.array(net.cell_response.ca[0][gid_oi][section])
        ax5.plot(net.cell_response.times, v + i*120, color=l23_cols_tone[i], linewidth=1.5)
        ax6.plot(net.cell_response.times, ca + i *1e-4, color=l23_cols_tone[i], linewidth=1.5)
        prevL += L[i]
    ax5.spines['left'].set_bounds([-70, 30])
    ax5.spines['bottom'].set_bounds([0, 200])
    ax5.set_xticks([])
    ax5.set_yticks([])
    ax5.spines['top'].set_visible(False)
    ax5.spines['right'].set_visible(False)

    ax6.spines['left'].set_bounds([5e-5, 7.5e-5])
    ax6.spines['bottom'].set_bounds([0, 200])
    ax6.set_xticks([])
    ax6.set_yticks([])
    ax6.spines['top'].set_visible(False)
    ax6.spines['right'].set_visible(False)

    U, C = np.unique(np.array(net.cell_response.spike_gids[0])[(np.array(net.cell_response.spike_gids[0]) < 270) & (np.array(net.cell_response.spike_gids[0]) >= 170)], return_counts=True)
    gid_oi = U[np.argmax(C)]

    sections = ['soma', 'apical_trunk', 'apical_1', 'apical_2', 'apical_tuft']
    l5_cols_tone = ['k','#2A3760','#36477B','#425796', '#5C72B7']

    L = []
    for section in sections:
        L.append(net.cell_types["L5_pyramidal"]['cell_object'].sections[section].L)
    prevL = 0
    for i, section in enumerate(sections):

        v = np.array(net.cell_response.vsec[0][gid_oi][section])
        ca = np.array(net.cell_response.ca[0][gid_oi][section])
        ax7.plot(net.cell_response.times, v + i*120, color=l5_cols_tone[i], linewidth=1.5)
        ax8.plot(net.cell_response.times, ca + i *1e-4, color=l5_cols_tone[i], linewidth=1.5)
        prevL += L[i]
    ax7.spines['left'].set_bounds([-70, 30])
    ax7.spines['bottom'].set_bounds([0, 200])
    ax7.set_xticks([])
    ax7.set_yticks([])
    ax7.spines['top'].set_visible(False)
    ax7.spines['right'].set_visible(False)

    ax8.spines['left'].set_bounds([5e-5, 7.5e-5])
    ax8.spines['bottom'].set_bounds([0, 200])
    ax8.set_xticks([])
    ax8.set_yticks([])
    ax8.spines['top'].set_visible(False)
    ax8.spines['right'].set_visible(False)


    return fig




def plot_multi_trial_sim(net, dpl):

    fig = plt.figure(constrained_layout=True, figsize=(20,4))

    gs = fig.add_gridspec(4,4, height_ratios=[.25, 0.25, 0.25, 0.25], hspace=0.3, wspace=.4)
    ax00 = fig.add_subplot(gs[0,0])
    ax10 = fig.add_subplot(gs[1:4,0])
    ax11 = fig.add_subplot(gs[0:2,1])
    ax21 = fig.add_subplot(gs[2:,1])
    ax3 = fig.add_subplot(gs[:,2])
    ax4 = fig.add_subplot(gs[:,3])

    # drives
    bins = np.arange(0, 1000, 5)
    spike_prox1 = np.array(net.cell_response.spike_times_by_type['prox_poiss'][0]).reshape(-1,1)

    ax00.hist(spike_prox1, bins=bins, color='r')
    ax00.set_xlim([0, 1000])
    ax00.set_xticklabels([])  
    ax00.set_xticks(np.arange(0, 1001, 250))
          
    # dipole
    ax10.plot(dpl[0].times, dpl[0].data['agg'], color='k', linewidth=1)
    ax10.set_ylabel('diple (nA·m)')
    ax10.set_xlim([0, 1000])
    ax10.set_xticks(np.arange(0, 1001, 250))
    ax10.spines['top'].set_visible(False)
    ax10.spines['right'].set_visible(False)
    ax10.set_xlabel('time (ms)')

    # dipole and raster
    I = np.array(net.cell_response.spike_gids[0]) < 135
    gids = np.array(net.cell_response.spike_gids[0]).reshape(-1,)[I]
    times = np.array(net.cell_response.spike_times[0]).reshape(-1,)[I] 
    colors = [assign_color(net, gid) for gid in gids]

    ax11.plot(dpl[0].times, dpl[0].data['L2'], color='k', alpha=0.25)
    ax11.set_ylim([0, 0.01])
    ax11.set_xlim([0, 1000])
    ax11.set_xticks(np.arange(0, 1001, 250))
    
    ax11.set_ylabel('dipole (nA·m)')
    ax11.set_yticks([0.0, 0.005, 0.01])
    ax12 = ax11.twinx()

    ax12.scatter(times, gids, s=15, alpha=1., c=colors, marker='|')
    ax12.set_ylabel('L2/3 neurons')
    ax12.set_xticklabels([])
    ax12.invert_yaxis()
    ax12.set_yticks([0, 35, 135])
    ax12.set_xticks(np.arange(0, 1001, 250))


    I = (np.array(net.cell_response.spike_gids[0]) >= 135) & (np.array(net.cell_response.spike_gids[0]) < 270)
    gids = np.array(net.cell_response.spike_gids[0]).reshape(-1,)[I]
    times = np.array(net.cell_response.spike_times[0]).reshape(-1,)[I] 
    colors = [assign_color(net, gid) for gid in gids]

    ax21.plot(dpl[0].times, dpl[0].data['L5'], color='k', alpha=0.25)
    ax21.set_xlim([0, 1000])
    ax21.set_xticks(np.arange(0, 1001, 250))

    ax21.set_ylim([-0.015, 0.03])
    ax21.set_yticks([-0.015, 0.0, 0.015, 0.03])
    ax22 = ax21.twinx()
    ax22.scatter(times, gids, s=15, alpha=1., c=colors, marker='|')
    ax22.set_ylabel('L5 neurons')
    ax22.invert_yaxis()
    ax22.set_yticks([135, 170, 270])
    ax21.set_xlabel('time (ms)')
    ax22.set_xticks(np.arange(0, 1001, 250))

    # TFR
    freqs = np.arange(20, 81, 1)

    cur_tfr = tfr_array_morlet(dpl[0].data["agg"][None, None, :], dpl[0].sfreq, freqs, output='power')
    tfr = cur_tfr.squeeze()
    
    im = ax3.pcolormesh(dpl[0].times, freqs, tfr, shading='auto', cmap='RdBu_r', vmin=0, vmax=np.round(tfr.max(),2), rasterized=True)
    cb = plt.colorbar(im, pad=0.025, orientation='horizontal', shrink=0.4)
    cb.set_ticks([0, np.round(tfr.max(),2)])
    cb.set_label('power (nA·m)^2')
    ax3.set_xlabel('time (ms)')
    ax3.set_yticks([20, 40, 60, 80])
    ax3.set_ylabel('frequency (Hz)')
    ax3.set_xticks(np.arange(0, 1001, 250))


    # average over trials

    tfr = np.zeros((len(dpl), len(freqs), len(dpl[0].times)))

    for i,d in enumerate(dpl):

        cur_tfr = tfr_array_morlet(d.data["agg"][None, None, :], d.sfreq, freqs, output='power')
        tfr[i] = cur_tfr.squeeze()

    tfr = tfr.mean(axis=0)
    
    im = ax4.pcolormesh(dpl[0].times, freqs, tfr, shading='auto', cmap='RdBu_r', vmin=0, vmax=np.round(tfr.max()-0.03,2), rasterized=True)
    cb = plt.colorbar(im, pad=0.025, orientation='horizontal', shrink=0.4)
    cb.set_ticks([0, np.round(tfr.max()-0.03,2)])
    cb.set_label('power (nA·m)^2')
    ax4.set_xlabel('time (ms)')
    ax4.set_yticks([20, 40, 60, 80])
    ax4.set_ylabel('frequency (Hz)')
    ax4.set_xticks(np.arange(0, 1001, 250))


    return fig

In [4]:
hnn_core_root = op.dirname(hnn_core.__file__)
params_fname = op.join(hnn_core_root, 'param', 'gamma_human_ET50Hz_sustained.json')
params = hnn_core.read_params(params_fname)

net = duecker_ET_model(params)

weights_ampa = {'L2_pyramidal': 0.0045, 'L5_pyramidal': 0.0055}
weights_nmda = {'L2_pyramidal': 0.0045, 'L5_pyramidal': 0.0055}
synaptic_delays = {'L2_pyramidal': 0.1, 'L5_pyramidal': 1.0}
rate_constant = {'L2_pyramidal': 200.0, 'L5_pyramidal': 200.0}

net.add_poisson_drive('prox_poiss', tstart=0, tstop=1000, rate_constant=rate_constant, weights_ampa=weights_ampa, weights_nmda=weights_nmda,
    location='proximal', synaptic_delays=synaptic_delays,
    event_seed=1349)

#with JoblibBackend(n_jobs=n_procs):
dpl = simulate_dipole(net, tstop=1000,bsl_cor='duecker', n_trials=1, record_ca='all', record_vsec='all')



In [ ]:
fig = plot_single_trial_sim(net, dpl)

fig.savefig(op.join(pathfig, 'gamma_ET_model_sustained.png'))
fig.savefig(op.join(pathfig, 'gamma_ET_model_sustained.pdf'))

In [ ]:
# bursty: reduce inhibition and reduce frequency and NMDA of drive
hnn_core_root = op.dirname(hnn_core.__file__)
params_fname = op.join(hnn_core_root, 'param', 'gamma_human_ET50Hz_bursty.json')
params = hnn_core.read_params(params_fname)

net = duecker_ET_model(params)
weights_ampa = {'L2_pyramidal': 0.0035, 'L5_pyramidal': 0.0055}
weights_nmda = {'L2_pyramidal': 0.0025, 'L5_pyramidal': 0.0035}
synaptic_delays = {'L2_pyramidal': 0.1, 'L5_pyramidal': 1.0}
rate_constant = {'L2_pyramidal': 100.0, 'L5_pyramidal': 100.0}
net.add_poisson_drive('prox_poiss', tstart=0, tstop=1000, rate_constant=rate_constant, weights_ampa=weights_ampa, weights_nmda=weights_nmda,
    location='proximal', synaptic_delays=synaptic_delays,
    event_seed=1349)

with JoblibBackend(n_jobs=n_procs):
    dpl = simulate_dipole(net, tstop=1000,bsl_cor='duecker', n_trials=100)




In [ ]:
import pickle
with open(op.join(pathfig, 'net_gamma_bursty.pkl'), 'wb') as f:
    pickle.dump(net, f)

with open(op.join(pathfig, "dpl_gamma_bursty.pkl"), 'wb') as f:
    pickle.dump(dpl,f)

In [ ]:
import pickle
with open(op.join(pathfig, 'net_gamma_bursty.pkl'), 'rb') as f:
    net = pickle.load(f)

with open(op.join(pathfig, "dpl_gamma_bursty.pkl"), 'rb') as f:
    dpl = pickle.load(f)
    
fig = plot_multi_trial_sim(net, dpl)

fig.savefig(op.join(pathfig, 'gamma_ET_model_bursty.png'))
fig.savefig(op.join(pathfig, 'gamma_ET_model_bursty.pdf'))



In [ ]:
del net, dpl

In [ ]:
# # sustained gamma 50 Hz - Neymotin reproduce?
# hnn_core_root = op.dirname(hnn_core.__file__)
# params_fname = op.join(hnn_core_root, 'param', 'gamma_human_ET50Hz_sustained.json')
# params = hnn_core.read_params(params_fname)

# net = duecker_ET_model(params)
# weights_ampa = {'L2_pyramidal': 0.0035, 'L5_pyramidal': 0.0065}
# weights_nmda = {'L2_pyramidal': 0.0012, 'L5_pyramidal': 0.0032}
# synaptic_delays = {'L2_pyramidal': 0.1, 'L5_pyramidal': 1.0}
# rate_constant = {'L2_pyramidal': 4.0, 'L5_pyramidal': 5.0}
# net.add_poisson_drive('prox_poiss', tstart=0, tstop=1000, rate_constant=rate_constant, weights_ampa=weights_ampa, weights_nmda=weights_nmda,
#     location='proximal', synaptic_delays=synaptic_delays,
#     event_seed=1349)

# #with MPIBackend(n_procs=n_procs, mpi_cmd='mpiexec'):
# dpl = simulate_dipole(net, tstop=1000,bsl_cor='duecker', n_trials=1, record_ca='all', record_vsec='all')


# fig = plot_single_trial_sim(net, dpl)

# fig.savefig(op.join(pathfig, 'gamma_ET_model_bursty_neymotin.png'))
# fig.savefig(op.join(pathfig, 'gamma_ET_model_bursty_neymotin.pdf'))

In [ ]:
# Exploratory: duecker ET model in response to gamma drive
hnn_core_root = op.dirname(hnn_core.__file__)

net = duecker_ET_model()
weights_ampa = {'L2_pyramidal': 0.0035, 'L5_pyramidal': 0.0055}
weights_nmda = {'L2_pyramidal': 0.0025, 'L5_pyramidal': 0.0035}
synaptic_delays = {'L2_pyramidal': 0.1, 'L5_pyramidal': 1.0}
rate_constant = {'L2_pyramidal': 100.0, 'L5_pyramidal': 100.0}

net.add_poisson_drive('prox_poiss', tstart=0, tstop=1000, rate_constant=rate_constant, weights_ampa=weights_ampa,
    location='proximal', synaptic_delays=synaptic_delays,
    event_seed=1349)

#with MPIBackend(n_procs=n_procs, mpi_cmd='mpiexec'):

dpl = simulate_dipole(net, tstop=1000,bsl_cor='duecker', n_trials=1, record_vsec='all', record_ca='all')



In [ ]:
fig = plot_single_trial_sim(net, dpl)


In [ ]:
fig.savefig(op.join(pathfig, 'gamma_drive_dueckerET_model.png'))
fig.savefig(op.join(pathfig, 'gamma_drive_dueckerET_model.pdf'))